In [1]:
import random
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import re

from keras.layers import TextVectorization,Embedding,LSTM,Dense,Dropout
from keras.models import Sequential
from keras.optimizers import Adam

from sklearn.model_selection import train_test_split

2024-01-06 20:11:49.334035: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [3]:
df = pd.read_csv('imdb.csv')
df = df.sample(10000)

In [4]:
def clean(text):
    result = text.lower()
    result = re.sub(r'<[^>]*>', r' ', result)
    result = re.sub(r'[^a-z\s\']', r'', result)
    result = re.sub(r'\s+', r' ', result)
    return result

df['Cleaned'] = df['Review'].apply(clean)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(df['Cleaned'],df['Sentiment'],test_size=.15,random_state=seed)

In [6]:
max_tokens = 1000
output_sequence_length = 500

vectorizer = TextVectorization(max_tokens=max_tokens,output_sequence_length=output_sequence_length)
vectorizer.adapt(x_train.values)

In [7]:
model = Sequential([
    vectorizer,
    Embedding(input_dim=max_tokens, input_length=output_sequence_length, output_dim=32, mask_zero=True),
    LSTM(256),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization (TextVe  (None, 500)               0         
 ctorization)                                                    
                                                                 
 embedding (Embedding)       (None, 500, 32)           32000     
                                                                 
 lstm (LSTM)                 (None, 256)               295936    
                                                                 
 dense (Dense)               (None, 1)                 257       
                                                                 
Total params: 328193 (1.25 MB)
Trainable params: 328193 (1.25 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
optimizer = Adam(learning_rate=.001)
model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
history = model.fit(x_train,y_train,epochs=50,batch_size=1000,validation_split=.15).history

Epoch 1/50
8/8 [==============================] - 92s 11s/step - loss: 0.6926 - accuracy: 0.5003 - val_loss: 0.6908 - val_accuracy: 0.4973
Epoch 2/50
8/8 [==============================] - 82s 10s/step - loss: 0.7185 - accuracy: 0.5736 - val_loss: 0.6858 - val_accuracy: 0.4996
Epoch 3/50
8/8 [==============================] - 84s 10s/step - loss: 0.6851 - accuracy: 0.5080 - val_loss: 0.6885 - val_accuracy: 0.4973
Epoch 4/50
8/8 [==============================] - 88s 11s/step - loss: 0.6855 - accuracy: 0.5075 - val_loss: 0.6869 - val_accuracy: 0.4980
Epoch 5/50
8/8 [==============================] - 89s 11s/step - loss: 0.6834 - accuracy: 0.5128 - val_loss: 0.6841 - val_accuracy: 0.5161
Epoch 6/50
4/8 [==============>...............] - ETA: 52s - loss: 0.6799 - accuracy: 0.5330 

In [ ]:
model.evaluate(x_test,y_test)

In [ ]:
plt.plot(history['loss'])
plt.plot(history['val_loss'])
plt.show()

In [ ]:
plt.plot(history['accuracy'])
plt.plot(history['val_accuracy'])
plt.show()

In [ ]:
predictions = model.predict(x_test)
testing = pd.DataFrame(y_test.values,index=y_test.index, columns=['Actual'])
testing['Prediction'] = predictions
testing['Rounded'] = testing['Prediction'].round().astype(int)

In [ ]:
pd.crosstab(testing['Actual'], testing['Rounded'], rownames=['Actual'], colnames=['Predicted'])

In [ ]:
plt.scatter(y_test,predictions)
plt.xlim(0,1)
plt.ylim(0,1)
plt.show()